# Microbiome Disease Intelligence Platform
Advanced starter notebook for microbiota disease analysis.

In [ ]:

# Microbiome Disease Intelligence Platform
# Complete Starter Notebook

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# -------------------------------
# Load Dataset
# -------------------------------
# Replace with your microbiome dataset
# df = pd.read_csv("microbiome.csv")

# Demo synthetic dataset
np.random.seed(42)

n = 300

df = pd.DataFrame({
    "Disease": np.random.choice(["Healthy","Diabetes","IBD"], n),
    "Bacteroides": np.random.rand(n)*100,
    "Prevotella": np.random.rand(n)*100,
    "Faecalibacterium": np.random.rand(n)*100,
    "Roseburia": np.random.rand(n)*100,
    "Akkermansia": np.random.rand(n)*100
})

print(df.head())

# -------------------------------
# Diversity Score
# -------------------------------
species_cols = df.columns[1:]

def shannon(row):
    p = row / row.sum()
    p = p[p > 0]
    return -(p*np.log(p)).sum()

df["Shannon"] = df[species_cols].apply(shannon, axis=1)

print(df["Shannon"].describe())

# -------------------------------
# Visualization
# -------------------------------
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Disease", y="Shannon")
plt.title("Alpha Diversity")
plt.show()

# -------------------------------
# PCA
# -------------------------------
X = df[species_cols]

pca = PCA(n_components=2)
pcs = pca.fit_transform(X)

pca_df = pd.DataFrame(pcs, columns=["PC1","PC2"])
pca_df["Disease"] = df["Disease"]

plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Disease")
plt.title("PCA of Microbiome Profiles")
plt.show()

# -------------------------------
# Disease Prediction
# -------------------------------
le = LabelEncoder()

y = le.fit_transform(df["Disease"])

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = RandomForestClassifier(n_estimators=200)
model.fit(X_train,y_train)

pred = model.predict(X_test)

print(classification_report(y_test,pred))

# -------------------------------
# Feature Importance
# -------------------------------
imp = pd.DataFrame({
    "Species": species_cols,
    "Importance": model.feature_importances_
}).sort_values("Importance",ascending=False)

print(imp)

plt.figure(figsize=(8,5))
sns.barplot(data=imp,x="Importance",y="Species")
plt.title("Important Microbes")
plt.show()

# -------------------------------
# Correlation Heatmap
# -------------------------------
plt.figure(figsize=(8,6))
sns.heatmap(df[species_cols].corr(),annot=True)
plt.title("Microbial Correlation Network")
plt.show()
